# Tabular Machine Learning Models for Walmart Sales Forecasting

This notebook trains, evaluates, and compares nine different machine learning models and neural networks on the Walmart sales forecasting task. We import modular training functions from the `src/train_models.py` package and evaluate all models using the standard validation split to ensure direct comparison with our baseline and statistical models.

## Models Implemented:
1. **Linear Regression** (OLS)
2. **Random Forest Regressor** (Bagging ensemble of decision trees)
3. **K-Nearest Neighbors (KNN)**
4. **XGBoost Regressor** (Gradient boosted trees with regularization)
5. **LightGBM Regressor** (Leaf-wise histogram gradient boosted trees)
6. **Multi-Layer Perceptron (MLP)** (Scikit-Learn Multi-Layer Perceptron)
7. **Artificial Neural Network (ANN)** (PyTorch feed-forward network with linear & ReLU layers)
8. **Gradient Boosted Regression Trees (GBRT)** (Scikit-Learn Gradient Boosting Regressor)
9. **Ensemble Regressor** (Simple average of Random Forest + XGBoost + LightGBM + MLP predictions)

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import time
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Append project root
sys.path.append(os.path.abspath(".."))

from src import config
from src.evaluate_models import (
    get_train_val_split,
    evaluate_predictions,
    calculate_wmae,
    calculate_mae
)
from src.train_models import (
    get_lr_forecast,
    get_rf_forecast,
    get_knn_forecast,
    get_xgb_forecast,
    get_lgb_forecast,
    get_mlp_forecast,
    get_ann_forecast,
    get_gbrt_forecast,
    get_ml_ensemble_forecast
)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

np.random.seed(42)

## 1. Load and Clean Tabular Features

We load the engineered features from `data/processed/train_features.csv` and drop rows containing NaNs (resulting from initial lag offsets).

In [ ]:
# Load engineered features
features_path = Path("..") / config.TRAIN_FEATURES_PATH.relative_to(config.BASE_DIR)
df = pd.read_csv(features_path)

# Standardize IsHoliday to boolean
if df["IsHoliday"].dtype == "object":
    df["IsHoliday"] = (
        df["IsHoliday"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map({"true": True, "false": False, "1": True, "0": False})
    )
df["IsHoliday"] = df["IsHoliday"].astype(bool)

df["Date"] = pd.to_datetime(df["Date"])

# Define features list
feature_cols = [
    'Store', 'Dept', 'IsHoliday', 'Size', 'Temperature', 'Fuel_Price',
    'CPI', 'Unemployment', 'Year', 'Month', 'WeekOfYear', 'Quarter',
    'Is_Q4', 'Is_Holiday_Month', 'Is_SuperBowl', 'Is_LaborDay', 'Is_Thanksgiving', 'Is_Christmas',
    'MarkDown_Total', 'Has_MarkDown', 'MarkDown_Avg', 'MarkDown_Max',
    'Weekly_Sales_lag_1', 'Weekly_Sales_lag_2', 'Weekly_Sales_lag_3', 'Weekly_Sales_lag_4',
    'Weekly_Sales_lag_8', 'Weekly_Sales_lag_12', 'Weekly_Sales_lag_26', 'Weekly_Sales_lag_52',
    'rolling_mean_4', 'rolling_mean_12', 'rolling_mean_26', 'rolling_mean_52',
    'rolling_std_4', 'rolling_std_12', 'rolling_std_52',
    'Store_Avg_Sales', 'Dept_Avg_Sales', 'Store_Dept_Avg_Sales',
    'Type_Encoded', 'Holiday_MarkDown', 'Q4_Holiday', 'Size_MarkDown'
]

# Drop rows with missing values in target or features
df_clean = df.dropna(subset=['Weekly_Sales'] + feature_cols).copy()

print(f"Original rows: {len(df):,}")
print(f"Cleaned rows:  {len(df_clean):,}")

## 2. Train / Validation Split

We split the cleaned dataset using the standard train/validation split date `2012-08-10`.

In [ ]:
# Split into train and validation sets
train_df = df_clean[df_clean["Date"] <= "2012-08-10"].copy()
val_df = df_clean[df_clean["Date"] > "2012-08-10"].copy()

y_val = val_df["Weekly_Sales"]

print(f"Training rows:   {len(train_df):,}")
print(f"Validation rows: {len(val_df):,}")

## 3. Run Tabular Machine Learning Models

We execute all nine models using the custom forecasting functions defined inside the `src/train_models.py` library.

In [ ]:
results_dict = {}
metrics_list = []

models = {
    "Linear Regression": lambda t, v, cols: get_lr_forecast(t, v, cols),
    "Random Forest": lambda t, v, cols: get_rf_forecast(t, v, cols),
    "KNN": lambda t, v, cols: get_knn_forecast(t, v, cols),
    "XGBoost": lambda t, v, cols: get_xgb_forecast(t, v, cols),
    "LightGBM": lambda t, v, cols: get_lgb_forecast(t, v, cols),
    "MLP": lambda t, v, cols: get_mlp_forecast(t, v, cols),
    "ANN": lambda t, v, cols: get_ann_forecast(t, v, cols),
    "GBRT": lambda t, v, cols: get_gbrt_forecast(t, v, cols),
    "Ensemble": lambda t, v, cols: get_ml_ensemble_forecast(t, v, cols)
}

for name, forecast_func in models.items():
    print(f"\n--- Running {name} ---")
    start_time = time.time()
    
    preds = forecast_func(train_df, val_df, feature_cols)
    runtime = time.time() - start_time
    
    col_name = f"{name}_Pred"
    val_df[col_name] = preds
    
    metrics = evaluate_predictions(
        y_true=y_val,
        y_pred=preds,
        is_holiday=val_df["IsHoliday"]
    )
    metrics["Model"] = name
    metrics["Runtime (s)"] = runtime
    metrics_list.append(metrics)
    
    print(f"  Completed in {runtime:.2f} seconds. WMAE: {metrics['WMAE']:,.2f}")

## 4. Compare Model Error Metrics

We compile error metrics across all models and display them in a ranked summary table.

In [ ]:
metrics_df = pd.DataFrame(metrics_list)[["Model", "WMAE", "MAE", "RMSE", "MAPE", "sMAPE", "Runtime (s)"]]
metrics_df = metrics_df.sort_values("WMAE").reset_index(drop=True)
display(metrics_df)

## 5. Visualizing Model Error Comparisons

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics_plot = ["WMAE", "MAE", "RMSE"]
colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]

for idx, metric in enumerate(metrics_plot):
    sns.barplot(data=metrics_df, x="Model", y=metric, ax=axes[idx], color=colors[idx])
    axes[idx].set_title(f"{metric} Comparison", fontsize=13, fontweight="bold")
    axes[idx].tick_params(axis='x', rotation=45)
    axes[idx].set_ylabel(metric)
    axes[idx].set_xlabel("")
    for p in axes[idx].patches:
        height = p.get_height()
        axes[idx].annotate(f'{height:,.1f}',
                    xy=(p.get_x() + p.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=9)

plt.suptitle("Machine Learning Models Error Comparison (All Groups)", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

## 6. Actual vs Predicted Weekly Sales

We plot predicted vs actual sales across a subset of validation dates to visualize alignment.

In [ ]:
agg_sales = val_df.groupby("Date")["Weekly_Sales"].sum().reset_index()

plt.figure(figsize=(14, 6))
plt.plot(agg_sales["Date"], agg_sales["Weekly_Sales"], label="Actual Sales", color="black", linewidth=3.0, marker="o")

top_models = metrics_df.head(3)["Model"].tolist()
for name in top_models:
    col_name = f"{name}_Pred"
    pred_agg = val_df.groupby("Date")[col_name].sum().reset_index()
    plt.plot(pred_agg["Date"], pred_agg[col_name], label=name, linestyle="--", marker="x", alpha=0.8)

plt.title("Aggregate Weekly Forecast vs Actual Sales (Top 3 Machine Learning Models)", fontsize=15, fontweight="bold")
plt.xlabel("Date", fontsize=12)
plt.ylabel("Total Sales ($)", fontsize=12)
plt.legend(loc="upper right", frameon=True)
plt.tight_layout()
plt.show()

## 7. Actual vs Predicted Scatter Analysis

We construct scatter plots of predicted vs actual sales capped at the 99th percentile of sales values.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(18, 18))
axes = axes.flatten()

plot_limit = max(
    val_df["Weekly_Sales"].quantile(0.99),
    val_df[[f"{name}_Pred" for name in models.keys()]].quantile(0.99).max()
)

for idx, name in enumerate(models.keys()):
    col_name = f"{name}_Pred"
    ax = axes[idx]
    
    ax.scatter(val_df["Weekly_Sales"], val_df[col_name], alpha=0.3, color="#4c72b0", edgecolors="w")
    ax.plot([0, plot_limit], [0, plot_limit], color="red", linestyle="--", linewidth=2)
    ax.set_xlim(0, plot_limit)
    ax.set_ylim(0, plot_limit)
    
    corr = val_df["Weekly_Sales"].corr(val_df[col_name])
    model_wmae = metrics_df.set_index("Model").loc[name, "WMAE"]
    
    ax.set_title(f"{name}\nr = {corr:.3f}, WMAE = {model_wmae:,.0f}", fontsize=12, fontweight="bold")
    ax.set_xlabel("Actual Weekly Sales ($)")
    ax.set_ylabel("Predicted Weekly Sales ($)")
    
plt.suptitle("Actual vs Predicted Weekly Sales Scatter Analysis (Capped at 99th Percentile)", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

## 8. Save Model Predictions and Metrics

We serialize the validation set predictions and output metrics to files.

In [ ]:
output_dir = Path("..") / "results"
output_dir.mkdir(parents=True, exist_ok=True)

# Save predictions
pred_path = output_dir / "ml_predictions.csv"
val_df.to_csv(pred_path, index=False)
print(f"Predictions saved successfully to: {pred_path.resolve()}")

# Save metrics
metrics_path = output_dir / "ml_metrics.json"
metrics_dict = metrics_df.set_index("Model").to_dict(orient="index")
with open(metrics_path, "w") as f:
    json.dump(metrics_dict, f, indent=4)
print(f"Metrics saved successfully to: {metrics_path.resolve()}")

## 9. Key Findings & Conclusions

In [ ]:
best_row = metrics_df.iloc[0]
print("=== SUMMARY OF KEY FINDINGS ===")
print(f"Best Performing Machine Learning Model: {best_row['Model']}")
print(f"Validation WMAE:                        {best_row['WMAE']:,.2f}")